# CartPole: Aprendiendo a Equilibrar con Q-Learning

## Que aprendemos aqui?

1. **Estados continuos**: cuando los datos no son discretos
2. **Discretizacion**: como convertir numeros reales a estados discretos
3. **Gymnasium**: entorno de simulacion para RL (reemplazo moderno de OpenAI Gym)
4. **Hiperparametros**: alpha, gamma, epsilon
5. **Exploracion vs Explotacion**: el balance clave
6. **Entrenamiento**: como guardar el mejor modelo
7. **Evaluacion**: ver el modelo entrenado en accion

## Introduccion

En la Leccion 1, resolvimos el problema de Pedro y el Lobo con **estados discretos** (posiciones en un tablero de 8x8).

Ahora vamos a un problema mas realista: **CartPole**. El objetivo es equilibrar un palo vertical sobre un carrito que se mueve.

**Por que es diferente?**
- En Pedro y el Lobo: 64 estados posibles (8x8)
- En CartPole: infinitos estados (posiciones y velocidades son numeros reales)

**Solucion**: Discretizar los estados.

## Cargar librerias

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import random

## Gymnasium: Que es?

**Gymnasium** (antes OpenAI Gym) es una libreria que provee entornos de simulacion para entrenar agentes RL. Tiene desde juegos simples hasta control de robots.

> **Nota**: OpenAI Gym fue descontinuado en 2022. Gymnasium es su reemplazo mantenido con API mejorada.

**CartPole**:
- Un carrito horizontal que se mueve izq/der
- Un palo vertical encima del carrito
- Objetivo: mantener el palo vertical el mayor tiempo posible
- Recompensa: +1 por cada paso que el palo se mantiene vertical
- Termino: si el palo se cae o el carrito se sale del area

### Inicializar el entorno

In [ ]:
env = gym.make("CartPole-v1")
print("Espacio de acciones:", env.action_space)
print("Espacio de observaciones:", env.observation_space)
print("Accion aleatoria:", env.action_space.sample())

### El espacio de observaciones

El entorno nos devuelve 4 valores en cada paso:

| Indice | Valor | Rango tipico |
|--------|-------|--------------|
| 0 | Posicion del carrito | -2.4 a 2.4 |
| 1 | Velocidad del carrito | -Inf a Inf |
| 2 | Angulo del palo | -0.21 a 0.21 |
| 3 | Velocidad angular del palo | -Inf a Inf |

**Problema**: 2 de 4 valores no tienen limites. No podemos hacer una Q-Table infinita.
**Solucion**: Discretizar (convertir a rangos fijos).

### Probar el entorno sin aprender nada

En Gymnasium, `reset()` devuelve `(obs, info)` y `step()` devuelve `(obs, reward, terminated, truncated, info)`.

In [ ]:
obs, info = env.reset()
done = False
total_reward = 0
while not done:
    obs, rew, terminated, truncated, info = env.step(env.action_space.sample())
    done = terminated or truncated
    total_reward += rew

print(f"Recompensa total (accion aleatoria): {total_reward}")
env.close()

Con accion aleatoria, el palo cae en ~10-20 pasos. Queremos llegar a 200+ pasos.

## Discretizacion de Estados

Necesitamos convertir los 4 valores continuos a un **estado discreto** (enteros).

### Metodo: Division por escalares

Dividimos cada valor por un factor y redondeamos a entero.

In [ ]:
def discretize(x):
    return tuple((x / np.array([0.25, 0.25, 0.01, 0.1])).astype(int))

# Ejemplo
obs, info = env.reset()
obs, _, _, _, _ = env.step(env.action_space.sample())
print("Observacion original:", obs)
print("Estado discretizado:", discretize(obs))

### Metodo alternativo: Bins (contenedores)

Otra forma es dividir cada rango en N contenedores.

In [ ]:
def create_bins(i, num):
    return np.arange(num + 1) * (i[1] - i[0]) / num + i[0]

print("Bins para (-5, 5) con 10 bins:", create_bins((-5, 5), 10))

ints = [(-5, 5), (-2, 2), (-0.5, 0.5), (-2, 2)]
nbins = [20, 20, 10, 10]
bins = [create_bins(ints[i], nbins[i]) for i in range(4)]

def discretize_bins(x):
    return tuple(np.digitize(x[i], bins[i]) for i in range(4))

## La Q-Table para estados continuos

En Pedro y el Lobo, la Q-Table era un tensor 8x8x4.

Para CartPole, usamos un **diccionario** porque no sabemos el tamano exacto del espacio de estados.

Clave del diccionario: `(estado, accion)`
Valor: valor Q correspondiente

In [ ]:
Q = {}
actions = (0, 1)  # 0=izquierda, 1=derecha

def qvalues(state):
    return [Q.get((state, a), 0) for a in actions]

## Hiperparametros

Los **hiperparametros** controlan como aprende el agente:

| Parametro | Que controla | Nuestro valor |
|-----------|-------------|---------------|
| `alpha` | Tasa de aprendizaje | 0.3 |
| `gamma` | Factor de descuento | 0.9 |
| `epsilon` | Balance exploracion/explotacion | 0.90 |

- **alpha**: Que tan rapido ajusta la Q-Table. Alto = aprende rapido pero es inestable.
- **gamma**: Importancia del futuro. 0.9 = le importa mucho el futuro.
- **epsilon**: % de veces que explora vs explota. 0.9 = explora 90% de las veces.

In [ ]:
alpha = 0.3
gamma = 0.9
epsilon = 0.90

## Algoritmo de Aprendizaje

### Mejoras respecto a la Leccion 1:

1. **Recompensa acumulada promedio**: cada 5000 epochs, promediamos las recompensas
2. **Guardamos el mejor modelo**: si el promedio mejora, guardamos la Q-Table

### Por que guardar el mejor modelo?

A veces el modelo "empeora" durante el entrenamiento. Guardamos la mejor version para usar despues.

In [ ]:
def probs(v, eps=1e-4):
    v = v - v.min() + eps
    v = v / v.sum()
    return v

Qmax = 0
cum_rewards = []
rewards = []

for epoch in range(100000):
    obs, info = env.reset()
    done = False
    cum_reward = 0

    while not done:
        s = discretize(obs)
        if random.random() < epsilon:
            v = probs(np.array(qvalues(s)))
            a = random.choices(actions, weights=v)[0]
        else:
            a = np.random.randint(env.action_space.n)

        obs, rew, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        cum_reward += rew
        ns = discretize(obs)
        Q[(s, a)] = (1 - alpha) * Q.get((s, a), 0) + alpha * (rew + gamma * max(qvalues(ns)))

    cum_rewards.append(cum_reward)
    rewards.append(cum_reward)

    if epoch % 5000 == 0:
        print(f"{epoch}: {np.average(cum_rewards)}, alpha={alpha}, epsilon={epsilon}")
        if np.average(cum_rewards) > Qmax:
            Qmax = np.average(cum_rewards)
            Qbest = Q.copy()
        cum_rewards = []

### Que observar en los resultados:

- **Cerca de 195**: El criterio oficial para "resolver" CartPole es 195 promedio en 100 ejecuciones
- **Recompensa que cae**: A veces el modelo "dania" lo que ya aprendio
- **Importancia de Qbest**: Guardamos la mejor version por si el modelo empeora despues

## Graficando el Progreso

In [ ]:
plt.plot(rewards)
plt.xlabel("Epoch")
plt.ylabel("Recompensa acumulada")
plt.title("Progreso crudo del entrenamiento")
plt.show()

La grafica cruda es muy ruidosa. Usemos un **promedio movil** para ver la tendencia.

In [ ]:
def running_average(x, window):
    return np.convolve(x, np.ones(window) / window, mode='valid')

plt.plot(running_average(rewards, 100))
plt.xlabel("Epoch")
plt.ylabel("Recompensa promedio (100 pasos)")
plt.title("Progreso del entrenamiento (suavizado)")
plt.show()

## Viendo el Resultado en Accion

Probemos el modelo entrenado. Usamos la misma estrategia de muestreo que durante el entrenamiento.

In [ ]:
obs, info = env.reset()
done = False
while not done:
    s = discretize(obs)
    v = probs(np.array(qvalues(s)))
    a = random.choices(actions, weights=v)[0]
    obs, _, terminated, truncated, _ = env.step(a)
    done = terminated or truncated

env.close()
print("El palo se mantuvo en pie!")

## Mejorando los Hiperparametros

### Que podemos ajustar?

1. **Disminuir epsilon gradualmente**: Empezar explorando mucho, despues explotar mas
2. **Ajustar alpha**: Empezar con alpha alto, despues bajarlo
3. **Probar diferentes bins**: Mas bins = mas precision pero mas estados

### Tareas para experimentar:

> **Tarea 1**: Juega con los valores de alpha, gamma y epsilon. Puedes llegar a 195?

> **Tarea 2**: Mide formalmente si resuelves el problema (195 promedio en 100 ejecuciones)

## Conceptos Clave

| Concepto | Que es | Diferencia con Leccion 1 |
|----------|--------|--------------------------|
| **Estado continuo** | Valores reales (no enteros) | Leccion 1: 64 estados fijos |
| **Discretizacion** | Convertir reales a enteros | Dividir por escalares o bins |
| **Q-Table como diccionario** | Clave=(estado,accion) | Leccion 1: tensor numpy |
| **Epsilon** | Balance explore/exploit | Nuevo concepto |
| **Qbest** | Mejor modelo encontrado | Nuevo concepto |
| **Promedio movil** | Suavizar graficas ruidosas | Nuevo concepto |

## Errores Comunes

| Error | Consecuencia |
|-------|--------------|
| Usar gym en vez de gymnasium | Incompatibilidad con numpy 2.x |
| Olvidar `done = terminated or truncated` | El loop no termina |
| epsilon muy bajo | No explora, se queda en optima local |
| epsilon muy alto | No aprovecha lo aprendido |
| alpha muy alto | Aprendizaje inestable |
| Sin Qbest | Guardas un modelo que no es el mejor |
| Discretizacion muy gruesa | No distingue estados similares |
| Discretizacion muy fina | Demasiados estados, Q-Table gigante |

## Siguiente paso

Has completado la seccion de **Aprendizaje por Refuerzo**! Ahora sabes:

1. Como un agente aprende por ensayo y error
2. Q-Learning y la ecuacion de Bellman
3. Discretizar estados continuos
4. Balancear exploracion y explotacion

**Proyectos futuros**: Redes neuronales para RL (Deep Q-Networks), Policy Gradient, y mas algoritmos avanzados.